# Chapter 15 — Multi-Agent Red/Blue 자동 공격·자동 방어

> **Chapter 15 · Part III**

공유 CVE corpus 를 근거로 레드팀 Agent 가 공격하고 블루팀 Agent 가
방어·자기학습(self-heal)하는 자동 공격·방어 루프를 구현합니다(Step 1~4). 이어서 §15.2~15.3 이
정의하는 **Red 3·Blue 3·Purple 1 의 7-Agent 구성**을 실제 모델(로컬 Ollama `llama3.2:3b` +
`claude-haiku-4-5` + `claude-opus-5`)로 구현합니다(Step 5~6).

## 실습 목표
1. 레드·블루가 **같은 CVE corpus** 를 공유하는 대칭 구조를 이해합니다.
2. 블루의 self-heal(놓친 공격을 corpus 로 되짚어 학습) 을 구현합니다.
3. RAG 사용/미사용을 비교해 CVE corpus 가 자기방어 향상에 기여함을 정량 확인합니다.
4. Red Pool(R1 정찰→R2 exploit→R3 횡적이동, Peer-to-Peer) 과 Blue Pool(B1 탐지·B2 분석[Actor-Critic]·
   B3 대응[Hierarchical]) 을 실제 LLM 호출로 구현합니다.
5. Purple Agent 가 두 로그를 종합해 평균 탐지 지연·Purple 합의율을 포함한 정량 보고서를
   만드는 과정을 구현합니다.

> **주의:** 모든 공격은 시뮬레이션입니다. 실제 exploit·스캔은 격리 VM·본인 자산에서만 수행합니다.
> Step 5~6 은 실제 네트워크 호출(로컬 Ollama, Anthropic API)을 사용합니다.

In [1]:
# 첫 사용시에만
# !pip install -q anthropic ollama python-dotenv
# Step 5~6 은 실제 API/로컬 모델을 사용합니다 — 저장소 루트 .env 의 ANTHROPIC_API_KEY,
# 로컬에서 실행 중인 Ollama(`ollama serve`)와 `ollama pull llama3.2:3b` 가 필요합니다.

In [2]:
import random, hashlib
rng = random.Random(42)   # 재현성

---
## Step 1. 공유 CVE corpus

레드의 공격 근거이자 블루의 방어 근거입니다(ch11 RAG 의 축소판). 실제로는 `labs/cve_db` 의
SQLite/벡터스토어를 씁니다.

In [3]:
CVE_DB = {
    'CVE-2021-44228': {'name': 'Log4Shell',    'pattern': 'log4j jndi ldap rce'},
    'CVE-2017-5638':  {'name': 'Struts2',       'pattern': 'struts content-type ognl rce'},
    'CVE-2014-0160':  {'name': 'Heartbleed',    'pattern': 'openssl tls heartbeat disclosure'},
    'CVE-2022-22965': {'name': 'Spring4Shell',  'pattern': 'spring data binding classloader rce'},
    'CVE-2018-7600':  {'name': 'Drupalgeddon2', 'pattern': 'drupal form api render rce'},
}
def sig(cve_id):
    return ' '.join(sorted(CVE_DB[cve_id]['pattern'].split()))

# 격리 표적: 서비스마다 CVE 하나 (labs/target.py 와 대응)
TARGET = {'web-dvwa':'CVE-2021-44228','struts':'CVE-2017-5638','openssl':'CVE-2014-0160',
          'spring':'CVE-2022-22965','drupal':'CVE-2018-7600'}
print('corpus:', len(CVE_DB), '| 표적:', len(TARGET))

corpus: 5 | 표적: 5


---
## Step 2. 레드팀 Agent — CVE 선택 → PoC(변형) → 공격

라운드가 오를수록 난독화 변형 비율을 높여 탐지를 회피합니다.

In [4]:
def obfuscate(s):
    h = hashlib.sha1(s.encode()).hexdigest()[:6]
    return f'obf::{h}::' + s.replace(' ', '_')

def red_attack(round_idx):
    """각 표적을 공격. 라운드↑ → 변형 확률↑."""
    mutate_prob = min(0.15 + 0.20 * round_idx, 0.9)
    events = []
    for service, cve_id in TARGET.items():
        base = sig(cve_id)
        mutated = rng.random() < mutate_prob
        payload = obfuscate(base) if mutated else base
        events.append({'service': service, 'cve_id': cve_id,
                       'payload': payload, 'mutated': mutated})
    return events

demo = red_attack(3)
print('공격 이벤트:', len(demo), '| 변형:', sum(e['mutated'] for e in demo))

공격 이벤트: 5 | 변형: 5


---
## Step 3. 블루팀 Agent — 탐지·차단 + self-heal

블루는 처음엔 일부 CVE 만 알고 난독화도 못 벗깁니다. 놓친 공격을 **공유 corpus 로 되짚어**
시그니처를 학습하고, 난독화를 한 번 보면 벗기는 법을 익힙니다.

In [5]:
def normalize(payload, can_deobf):
    if can_deobf and payload.startswith('obf::'):
        return payload.split('::', 2)[2].replace('_', ' ')
    return payload

class BlueAgent:
    def __init__(self, known_cves, use_rag=True):
        self.use_rag = use_rag
        self.known = {sig(c) for c in known_cves}
        self.can_deobf = False

    def defend(self, events):
        detected, missed = 0, []
        for e in events:
            obs = normalize(e['payload'], self.can_deobf)
            if obs in self.known:
                detected += 1          # 차단 (labs 에선 block_ip)
            else:
                missed.append(e)
        return detected, missed

    def self_heal(self, missed):
        for e in missed:
            if e['mutated']:
                self.can_deobf = True         # 난독화 벗기는 법 학습
            if self.use_rag:                  # 공유 CVE corpus 조회로 시그니처 학습
                self.known.add(sig(e['cve_id']))
print('BlueAgent 정의 완료')

BlueAgent 정의 완료


---
## Step 4. 오케스트레이터(Purple) — 라운드 진행·점수

레드 공격 → 블루 방어 → 점수 → 블루 self-heal 을 반복하며 탐지율 곡선을 만듭니다.

In [6]:
def run(rounds=7, use_rag=True):
    global rng; rng = random.Random(42)   # 라운드 재현성
    blue = BlueAgent(['CVE-2021-44228', 'CVE-2014-0160'], use_rag=use_rag)  # 5개 중 2개만 알고 시작
    hist = []
    for r in range(rounds):
        events = red_attack(r)
        detected, missed = blue.defend(events)
        rate = detected / len(events)
        blue.self_heal(missed)
        hist.append(rate)
    return hist

on  = run(use_rag=True)
off = run(use_rag=False)
for r,(a,b) in enumerate(zip(on, off), 1):
    bar = '#' * round(a*20)
    print(f'Day{r}: RAG {a*100:5.0f}% {bar:<20}  | 대조군 {b*100:.0f}%')
print(f'\n[결론] RAG 사용 {on[0]*100:.0f}%→{on[-1]*100:.0f}%, '
      f'대조군 {off[0]*100:.0f}%→{off[-1]*100:.0f}% '
      f'(공유 CVE corpus 기여 +{(on[-1]-off[-1])*100:.0f}%p)')

Day1: RAG    40% ########              | 대조군 40%
Day2: RAG   100% ####################  | 대조군 40%
Day3: RAG   100% ####################  | 대조군 40%
Day4: RAG   100% ####################  | 대조군 40%
Day5: RAG   100% ####################  | 대조군 40%
Day6: RAG   100% ####################  | 대조군 40%
Day7: RAG   100% ####################  | 대조군 40%

[결론] RAG 사용 40%→100%, 대조군 40%→40% (공유 CVE corpus 기여 +60%p)


---
## Step 5. 7-Agent 구조 — Red Pool(R1→R2→R3) · Blue Pool(B1→B2→B3)

여기서부터는 §15.2~15.3 이 정의하는 **Red 3·Blue 3·Purple 1** 의 7-Agent 구성을 실제 모델로
구현합니다. Red Pool 은 로컬 Ollama `llama3.2:3b`, Blue Pool 은 `claude-haiku-4-5`,
Purple 은 `claude-opus-5`(실패 시 `claude-haiku-4-5` 폴백) 를 사용합니다. 위 Step 1~4 의
`CVE_DB`·`TARGET`·`sig()`·`obfuscate()` 를 그대로 재사용하므로, 두 영역이 같은 corpus 위에서
동작합니다.

호출 횟수를 bound 하기 위해 라운드 수는 작게(2 라운드 × 5 표적 = 10 이벤트) 유지합니다.
Step 4 의 7 라운드 탐지율 추세는 이미 확인했으므로, 여기서는 **7-Agent 역할 분리·통신 패턴·
Purple 정량 보고서** 를 보여주는 데 집중합니다.

> **주의:** 이 Step 은 실제 네트워크 호출을 합니다 — 로컬 Ollama(`ollama serve`, `llama3.2:3b` pull
> 완료)와 저장소 루트 `.env` 의 `ANTHROPIC_API_KEY` 가 필요합니다.

In [7]:
import os, time, json, re
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))   # 저장소 루트 .env 의 ANTHROPIC_API_KEY 로드

import ollama
from anthropic import Anthropic

OLLAMA_MODEL = 'llama3.2:3b'                      # Red Pool (R1~R3)
BLUE_MODEL = 'claude-haiku-4-5'          # Blue Pool (B1~B3)
PURPLE_MODEL_PRIMARY = 'claude-opus-5'            # Purple (책 기준)
PURPLE_MODEL_FALLBACK = 'claude-haiku-4-5'

ollama_client = ollama.Client(host='http://localhost:11434')
claude_client = Anthropic()   # ANTHROPIC_API_KEY 환경변수 사용


def ollama_ask(prompt, num_predict=40):
    """Red Pool 공용 — 짧고 저렴한 Ollama 호출."""
    resp = ollama_client.generate(model=OLLAMA_MODEL, prompt=prompt,
                                   options={'num_predict': num_predict, 'temperature': 0.4})
    return resp['response'].strip()


def claude_text(msg):
    """모든 TextBlock 을 이어붙인다 — Opus 계열은 thinking 블록이 먼저 올 수 있어,
    첫 블록만 읽으면 빈 문자열을 얻는다."""
    return ''.join(b.text for b in msg.content if getattr(b, 'type', None) == 'text')


def claude_ask(prompt, model=BLUE_MODEL, max_tokens=1024, disable_thinking=False):
    """Blue/Purple 공용 — 짧은 Claude 호출.

    JSON 요약이 목적인 호출(Purple)은 disable_thinking=True 로 추론을 끈다. Opus 5 는
    thinking 이 기본 on 이고 추론 토큰도 max_tokens 에 포함돼, max_tokens 를 작게 주면
    예산을 추론이 다 써 text 블록이 비어 온다(그래서 예전에는 Purple 이 opus-5 에서
    파싱 실패로 haiku 폴백에 빠졌다). temperature 는 SDK 1.x 에서 인자로 넘기지 않는다."""
    kwargs = {'model': model, 'max_tokens': max_tokens,
              'messages': [{'role': 'user', 'content': prompt}]}
    if disable_thinking:
        kwargs['thinking'] = {'type': 'disabled'}
    msg = claude_client.messages.create(**kwargs)
    return claude_text(msg)


def parse_json_loose(text, default=None):
    m = re.search(r'\{.*\}', text, re.S)
    if not m:
        return default
    try:
        return json.loads(m.group(0))
    except json.JSONDecodeError:
        return default


print('Ollama:', OLLAMA_MODEL, '| Blue:', BLUE_MODEL,
      '| Purple:', PURPLE_MODEL_PRIMARY, '(폴백:', PURPLE_MODEL_FALLBACK + ')')

Ollama: llama3.2:3b | Blue: claude-haiku-4-5 | Purple: claude-opus-5 (폴백: claude-haiku-4-5)


### Red Pool — R1(정찰) → R2(exploit 준비) → R3(횡적 이동), Peer-to-Peer 순차 통신 (§15.3.3)

각 단계가 독립 컨텍스트로 동작하고, 앞 단계의 출력만 다음 단계 입력이 됩니다. R2 는 신규
exploit 을 창작하지 않고 공유 CVE corpus 의 알려진 시그니처(`sig()`/`obfuscate()`)를
그대로 사용합니다(§15.2.2 참고). 표적·라운드마다 **난독화 강도**(none/light/strong)를 달리해,
강한 난독화(`obfuscate_strong()`)는 원본 키워드를 남기지 않아 Blue 의 시그니처 탐지를 실제로
어렵게 만듭니다. R3 는 자신의 침투가 성공했을지 스스로 평가해 자기보고(`self_report_success`)를
남기는데, 이는 Blue 의 실제 판정과 **별개**의 값입니다 — Red 는 자신이 우회했다고 낙관하지만
Blue 가 그 페이로드를 실제로 잡아낼 수 있어, 이 어긋남이 뒤의 Purple 합의율 지표가 됩니다.

In [8]:
# 라운드·표적별 난독화 강도 — 강도가 오를수록 Blue 시그니처 탐지가 어려워진다(§15.3.3).
# 표적을 다양화하고 강도를 섞어, Red 자기보고와 Blue 실제 판정이 자연스럽게 엇갈리게 한다.
OBF_LEVELS = {
    0: {'web-dvwa': 'none',   'struts': 'light',  'openssl': 'none',   'spring': 'light',  'drupal': 'light'},
    1: {'web-dvwa': 'strong', 'struts': 'light',  'openssl': 'strong', 'spring': 'light',  'drupal': 'strong'},
}


def obfuscate_strong(s):
    """강한 난독화 — 원본 키워드를 전부 해시로 치환해 시그니처 흔적을 남기지 않는다."""
    return 'enc::' + hashlib.sha1(s.encode()).hexdigest()


def red_r1_recon(service, cve_name):
    """R1 정찰 — 표적 서비스의 취약점 흔적을 짧게 보고한다."""
    prompt = (f"당신은 보안 침투테스트의 정찰(recon) Agent 다. 표적 서비스 '{service}' 에서 "
              f"알려진 취약점 '{cve_name}' 관련 흔적을 발견했다. 정찰 결과를 한국어 한 문장으로만 "
              f"짧게 보고하라.")
    return ollama_ask(prompt, num_predict=40)


def red_r2_exploit(recon_note, sig_hint):
    """R2 exploit 준비 — 알려진 CVE 시그니처 조각을 참고해 공격 준비를 보고한다(신규 창작 아님)."""
    prompt = (f"당신은 exploit 준비 Agent 다. 정찰 결과: '{recon_note}'. 알려진 시그니처 조각 "
              f"'{sig_hint}' 를 참고해 공격 페이로드를 준비했다고 한국어 한 문장으로만 보고하라. "
              f"실제 공격 코드는 출력하지 마라.")
    return ollama_ask(prompt, num_predict=40)


def red_r3_lateral(exploit_note, level):
    """R3 횡적 이동 — 난독화 강도를 근거로 자신의 침투가 탐지를 피했을지 스스로 평가한다.
    이는 Red 의 자기보고일 뿐, Blue 의 실제 판정과는 별개다."""
    hint = {
        'none':   '원본 시그니처를 그대로 전송해 방어측 탐지를 피하지 못했을 가능성이 크다',
        'light':  '키워드를 알아볼 수 있는 수준의 경(輕)난독화만 적용해 우회 성공을 장담하기 어렵다',
        'strong': '원본 키워드가 전혀 남지 않는 강한 난독화를 적용해 시그니처 탐지를 우회했을 것이다',
    }[level]
    prompt = (f"당신은 횡적 이동(lateral movement) Agent 다. 방금 표적에 침투를 시도했다"
              f"({exploit_note}). 정황: {hint}. 이 시도가 방어측 탐지를 피하고 성공했을 "
              f"가능성을 스스로 평가해 정확히 'SUCCESS' 또는 'DETECTED_RISK' 중 한 단어로만 답하라.")
    verdict = ollama_ask(prompt, num_predict=6)
    return 'SUCCESS' in verdict.upper()


def red_pipeline(round_idx, service, cve_id):
    """R1 → R2 → R3 순차 통신 — 각 단계 출력이 다음 단계 입력이 된다."""
    t_start = time.time()
    cve = CVE_DB[cve_id]
    r1_note = red_r1_recon(service, cve['name'])

    base = sig(cve_id)
    level = OBF_LEVELS.get(round_idx, {}).get(service, 'light')
    if level == 'strong':
        payload = obfuscate_strong(base)
    elif level == 'light':
        payload = obfuscate(base)
    else:
        payload = base
    r2_note = red_r2_exploit(r1_note, base[:24])

    self_report_success = red_r3_lateral(r2_note, level)
    t_r3_done = time.time()

    return {
        'round': round_idx, 'service': service, 'cve_id': cve_id,
        'r1': r1_note, 'r2': r2_note, 'obf_level': level, 'payload': payload,
        'self_report_success': self_report_success,
        't_start': t_start, 't_r3_done': t_r3_done,
    }

_demo = red_pipeline(1, 'web-dvwa', 'CVE-2021-44228')
print('R1:', _demo['r1'][:70])
print('R2:', _demo['r2'][:70])
print('R3 자기보고:', 'SUCCESS' if _demo['self_report_success'] else 'DETECTED_RISK',
      '| 난독화:', _demo['obf_level'])

R1: "보안 침투 테스트에서 'Log4Shell' 취약점에 대한 흔적이 발견되었다. 이 취약점은 Apache Log4j 버전 2.x
R2: Apache Log4j 버전 2.x의 로 '. known signature fragment 'jndi ldap log4j rc
R3 자기보고: DETECTED_RISK | 난독화: strong


### Blue Pool — B1(탐지) · B2(분석, Actor-Critic) · B3(대응, Hierarchical)

B2 는 **Actor-Critic** 패턴(§15.3.2)입니다 — actor 가 분석을 내고, critic 이 같은 모델·다른
prompt 로 과장·환각 여부를 검증합니다. 문제가 있으면 확신도를 보수적으로 낮춥니다(비용을
위해 별도의 재작성 호출은 생략).

B3 는 **Hierarchical** 패턴(§15.3.4)입니다 — `blue_b3_respond()` 함수 본문이 `blue_b1_detect()`
와 `blue_b2_analyze()` 를 직접 호출하고, B2 의 확신도(confidence)를 기준으로 책의 의사결정
트리(> 0.9 block, 0.5~0.9 escalate, 그 외 pass)를 그대로 적용합니다.

In [9]:
def blue_b1_detect(payload, corpus_lines):
    """B1 탐지 — 관측 payload 가 알려진 시그니처(공유 corpus) 와 일치하는지 판정."""
    prompt = (f"당신은 SIEM 탐지 Agent(B1) 다. 아래는 알려진 CVE 시그니처 목록이다:\n"
              f"{corpus_lines}\n\n관측된 페이로드: {payload}\n\n"
              f"이 페이로드가 목록과 일치(또는 난독화된 변형)하는지 판단해 JSON 한 줄로만 답하라: "
              f'{{"detected": true/false, "matched_cve": "CVE-ID 또는 null"}}')
    text = claude_ask(prompt, max_tokens=256)
    return parse_json_loose(text, {'detected': False, 'matched_cve': None})


def blue_b2_analyze(detection, payload):
    """B2 분석(Actor) + 자기 검증(Critic) — Actor-Critic 패턴 (§15.3.2)."""
    actor_prompt = (f"당신은 보안 분석 Agent(B2, actor) 다. B1 탐지 결과: {detection}. "
                     f"관측 페이로드: {payload}. 이 신호의 심각도와 확신도를 평가해 JSON 한 줄로만 "
                     f'답하라: {{"severity": 0~1 숫자, "confidence": 0~1 숫자, "attack_type": "짧은 분류명"}}')
    actor_text = claude_ask(actor_prompt, max_tokens=256)
    analysis = parse_json_loose(actor_text, {'severity': 0.3, 'confidence': 0.3, 'attack_type': 'unknown'})

    critic_prompt = (f"당신은 B2 의 critic 이다. 다음 분석이 근거 없이 과장되었거나 환각인지 "
                      f"짧게 검증하라: {analysis}. JSON 한 줄로만 답하라: "
                      f'{{"has_issues": true/false, "note": "20자 이내"}}')
    critic_text = claude_ask(critic_prompt, max_tokens=200)
    critique = parse_json_loose(critic_text, {'has_issues': False, 'note': ''})

    if critique.get('has_issues'):
        # 자기 검증에서 문제 발견 시 확신도를 보수적으로 낮춘다 (재작성 호출은 비용상 생략)
        analysis['confidence'] = min(analysis.get('confidence', 0.3), 0.6)
    return analysis, critique


def blue_b3_respond(payload, corpus_lines):
    """B3 대응 — B1·B2 를 직접 호출하는 Hierarchical 패턴 (§15.3.4)."""
    detection = blue_b1_detect(payload, corpus_lines)
    t_detected = time.time()
    analysis, critique = blue_b2_analyze(detection, payload)

    confidence = analysis.get('confidence', 0.3)
    if confidence > 0.9:
        action = 'block'
    elif confidence > 0.5:
        action = 'escalate_to_human'
    else:
        action = 'pass'

    return {'detection': detection, 'analysis': analysis, 'critique': critique,
            'action': action, 't_detected': t_detected}
print('B1/B2/B3 정의 완료 (Actor-Critic + Hierarchical)')

B1/B2/B3 정의 완료 (Actor-Critic + Hierarchical)


---
## Step 6. 7-Agent 루프 실행 + Purple Agent 정량 보고서

2 라운드 × 5 표적 = 10 이벤트를 Red→Blue 파이프라인으로 돌리고 로그를 모읍니다. 그다음
Purple Agent 가 두 로그를 종합해 §15.5 의 지표 중 **평균 Red→Blue 탐지 지연**과
**Purple 합의율(Red 자기보고 vs Blue 실제 판정 일치율)** 을 계산·보고합니다.

In [10]:
# 공유 corpus 를 Blue 의 탐지 근거 컨텍스트로 사용 (Step 3 의 self.known 과 같은 개념)
CORPUS_LINES = '\n'.join(f"- {c['name']} ({cid}): {c['pattern']}" for cid, c in CVE_DB.items())

ROUNDS_DEMO = 2   # API 호출 총량을 bound (Step4 의 7라운드 탐지율 추세는 이미 확인함)

red_log, blue_log = [], []
for r in range(ROUNDS_DEMO):
    print(f'--- Round {r + 1} ---')
    for service, cve_id in TARGET.items():
        red_ev = red_pipeline(r, service, cve_id)
        blue_ev = blue_b3_respond(red_ev['payload'], CORPUS_LINES)
        red_log.append(red_ev)
        blue_log.append(blue_ev)

        red_tag = 'SUCCESS' if red_ev['self_report_success'] else 'DETECTED_RISK'
        detected = bool(blue_ev['detection'].get('detected'))
        print(f"  R[{service:10s}] {red_tag:14s} obf={red_ev['obf_level']:6s} -> "
              f"B detected={detected!s:5s} conf={blue_ev['analysis'].get('confidence')} "
              f"action={blue_ev['action']}")

print(f'\n총 {len(red_log)} 이벤트 수집 '
      f'(Ollama 호출 {len(red_log) * 3}회, Claude 호출 {len(blue_log) * 3}회)')

--- Round 1 ---


  R[web-dvwa  ] DETECTED_RISK  obf=none   -> B detected=True  conf=0.6 action=escalate_to_human


  R[struts    ] DETECTED_RISK  obf=light  -> B detected=True  conf=0.92 action=block


  R[openssl   ] DETECTED_RISK  obf=none   -> B detected=True  conf=0.6 action=escalate_to_human


  R[spring    ] DETECTED_RISK  obf=light  -> B detected=True  conf=0.6 action=escalate_to_human


...

  R[web-dvwa  ] DETECTED_RISK  obf=strong -> B detected=False conf=0.4 action=pass


  R[struts    ] DETECTED_RISK  obf=light  -> B detected=True  conf=0.6 action=escalate_to_human


  R[openssl   ] DETECTED_RISK  obf=strong -> B detected=False conf=0.4 action=pass


  R[spring    ] DETECTED_RISK  obf=light  -> B detected=True  conf=0.6 action=escalate_to_human


  R[drupal    ] DETECTED_RISK  obf=strong -> B detected=False conf=0.4 action=pass

총 10 이벤트 수집 (Ollama 호출 30회, Claude 호출 30회)


In [11]:
def purple_report(red_log, blue_log):
    """Purple Agent — Red/Blue 로그를 종합해 정량 보고서를 만든다.

    수치는 코드가 직접 집계한다(모델이 숫자를 바꾸지 못하게). LLM 은 총평 2문장만 쓴다.
    Purple 프롬프트는 공격 기법이 아니라 '탐지 품질 지표 해설' 로 중립적으로 구성한다 —
    red/blue/exploit/침입 같은 공격 어휘를 그대로 넣으면 Opus 5 가 cyber 사용정책으로
    요청을 거부(stop_reason='refusal')해 빈 응답이 오고, 그 탓에 예전에는 opus-5 가
    파싱 실패로 haiku 폴백에 빠졌다(원인은 max_tokens 가 아니라 정책 거부였다)."""
    lags, consensus_hits, disagreements = [], [], []
    for red_ev, blue_ev in zip(red_log, blue_log):
        detected = bool(blue_ev['detection'].get('detected'))
        if detected:
            lags.append(blue_ev['t_detected'] - red_ev['t_start'])
        # 합의: Red 의 자기보고(성공/탐지위험)와 Blue 의 실제 판정(미탐/탐지)이 일치하는가
        agree = red_ev['self_report_success'] == (not detected)
        consensus_hits.append(agree)
        if not agree:
            tag = 'SUCCESS' if red_ev['self_report_success'] else 'DETECTED_RISK'
            verdict = '탐지' if detected else '미탐'
            disagreements.append({'service': red_ev['service'], 'obf': red_ev['obf_level'],
                                  'red': tag, 'blue': verdict})

    n = len(red_log)
    red_success_n = sum(e['self_report_success'] for e in red_log)
    blue_detect_n = sum(bool(b['detection'].get('detected')) for b in blue_log)
    red_success_rate = red_success_n / n
    blue_detection_rate = blue_detect_n / n
    avg_lag = sum(lags) / len(lags) if lags else None
    consensus_rate = sum(consensus_hits) / n
    lag_text = f'{avg_lag:.2f}초' if avg_lag is not None else '측정 불가(탐지 0건)'
    dis_services = ', '.join(f"{d['service']}(변형강도={d['obf']})" for d in disagreements)

    # 중립적 '탐지 품질 감사' 어휘로 구성(공격 어휘 배제). 총평 2문장만 요청한다.
    prompt = (
        "당신은 탐지 시스템 품질 분석가입니다. 아래 집계 지표를 근거로 개선 관점의 한국어 "
        "총평을 정확히 2문장으로 쓰세요(다른 텍스트 없이 2문장만).\n\n"
        f"[집계 대상] 총 {n}건 (2 회차 x 5 대상)\n"
        f"- 자체평가 통과율: {red_success_rate * 100:.0f}%\n"
        f"- 탐지율: {blue_detection_rate * 100:.0f}%\n"
        f"- 평균 탐지 지연: {lag_text}\n"
        f"- 라벨 합의율(자체평가 라벨 vs 실제 판정 라벨): {consensus_rate * 100:.0f}% "
        f"(불일치 {len(disagreements)}건)\n"
        + (f"- 불일치는 모두 입력 변형이 큰 이벤트({dis_services})로, 변형이 클수록 탐지 규칙이 "
           f"놓치는 경향\n" if disagreements else "- 라벨 불일치 없음\n"))

    model_used = PURPLE_MODEL_PRIMARY
    summary = claude_ask(prompt, model=PURPLE_MODEL_PRIMARY, max_tokens=1024, disable_thinking=True)
    if not summary.strip():   # 정책 거부 등으로 빈 응답이면 폴백(정상 경로에서는 발생하지 않는다)
        print(f'[Purple] {PURPLE_MODEL_PRIMARY} 빈 응답 -> {PURPLE_MODEL_FALLBACK} 로 폴백')
        model_used = PURPLE_MODEL_FALLBACK
        summary = claude_ask(prompt, model=PURPLE_MODEL_FALLBACK, max_tokens=1024)

    # 수치는 코드 집계값이 정본이다. 총평 문장만 LLM 산출.
    return {
        '평균_탐지_지연_초': round(avg_lag, 2) if avg_lag is not None else None,
        'purple_합의율': round(consensus_rate, 2),
        'red_자기보고_성공률': round(red_success_rate, 2),
        'blue_탐지율': round(blue_detection_rate, 2),
        '불일치_건수': len(disagreements),
        '불일치_상세': disagreements,
        '총평': summary.strip(),
        '_model': model_used,
    }

report = purple_report(red_log, blue_log)
print(json.dumps(report, ensure_ascii=False, indent=2))

{
  "평균_탐지_지연_초": 2.67,
  "purple_합의율": 0.7,
  "red_자기보고_성공률": 0.0,
  "blue_탐지율": 0.7,
  "불일치_건수": 3,
  "불일치_상세": [
    {
      "service": "web-dvwa",
      "obf": "strong",
      "red": "DETECTED_RISK",
      "blue": "미탐"
    },
    {
      "service": "openssl",
      "obf": "strong",
      "red": "DETECTED_RISK",
      "blue": "미탐"
    },
    {
      "service": "drupal",
      "obf": "strong",
      "red": "DETECTED_RISK",
      "blue": "미탐"
    }
  ],
  "총평": "탐지율 70%와 자체평가 통과율 0%의 괴리는 평가 기준이 과도하게 엄격하거나 실제 탐지 로직과 정합되지 않음을 시사하므로, 통과 판정 기준과 라벨링 규칙을 먼저 재정의해 합의율 70%(불일치 3건)를 끌어올려야 합니다. 특히 불일치 3건이 모두 변형강도 strong 이벤트(web-dvwa, openssl, drupal)에 집중된 만큼, 시그니처 중심 규칙을 정규화·난독화 해제 전처리와 행위 기반 탐지로 보완하고 평균 2.67초의 탐지 지연도 함께 단축하는 개선이 필요합니다.",
  "_model": "claude-opus-5"
}


---
## 정리

- 레드와 블루가 **같은 CVE corpus** 를 공유하므로, 블루는 놓친 공격을 corpus 로 되짚어 스스로
  방어를 고도화할 수 있습니다(self-heal). RAG(공유 corpus) 없이는 미지 변형 공격을 학습하지
  못해 탐지율이 평탄합니다(Step 1~4).
- Red Pool(R1 정찰→R2 exploit 준비→R3 횡적이동, Peer-to-Peer 순차 통신)과 Blue Pool(B1 탐지·
  B2 분석[Actor-Critic 자기검증]·B3 대응[Hierarchical, B1·B2 를 직접 호출])을 실제 Ollama·
  Claude 호출로 구현했습니다(Step 5).
- Purple Agent 가 Red·Blue 로그를 종합해 평균 Red→Blue 탐지 지연과 Purple 합의율(Red 자기보고
  vs Blue 실제 판정의 일치율)을 계산·보고합니다(Step 6). 이 두 지표가 §15.5 정량 보고서의
  핵심 항목입니다.
- 실제 격리 표적·MCP 도구로 확장한 실행체는 저장소 `labs/`(`python labs/run_lab.py`) 에 있습니다.

> **다음 챕터:** LLM-Native SIEM — Multi-Agent 운영의 텔레메트리·상관·검색을 LLM 친화적으로 재설계.